In [103]:
import os
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

In [104]:
print(torch.__version__)
print(transformers.__version__)

2.7.0+cu126
4.51.3


# 定义TaskVector

In [111]:
class TaskVector:
    def __init__(self, pretrained_model, finetuned_model):
        # task vector
        self.task_vector_param_dict = {}
        pretrained_param_dict = {param_name: param_value
                for param_name, param_value in pretrained_model.named_parameters()}
        finetuned_param_dict = {param_name: param_value
                for param_name, param_value in finetuned_model.named_parameters()}

        # 这里可以排除一些参数不做合并
        param_names_to_merge = list(pretrained_param_dict.keys())
        with torch.no_grad():
            for param_name in param_names_to_merge:
                self.task_vector_param_dict[param_name] = finetuned_param_dict[param_name] - pretrained_param_dict[param_name]

    def combine_with_pretrained_model(self, pretrained_model, scaling_coefficient=0.5):
        """模型参数的合并，求和
        """
        pretrained_param_dict = {param_name: param_value
                for param_name, param_value in pretrained_model.named_parameters()} 
        with torch.no_grad():
            merged_params = {}
            for param_name in self.task_vector_param_dict.keys():
                merged_params[param_name] = pretrained_param_dict[param_name] + scaling_coefficient * self.task_vector_param_dict[param_name]

        return merged_params

# 定义合并函数

In [112]:
def merging_models(merged_model, models_to_merge, scaling_coefficient):
    """合并模型参数
       merged_model: Base Model类
       models_to_merge： list[Model类]
    """

    merged_task_vector = None
    # 可以合并多个模型
    while len(models_to_merge) > 0:
        if merged_task_vector is None:
            merged_task_vector = TaskVector(
                pretrained_model=merged_model,
                finetuned_model=models_to_merge.pop(0)
            )
        else:
            merged_task_vector += TaskVector(
                pretrained_model=merged_model,
                finetuned_model=models_to_merge.pop(0)
            )
    # 执行合并
    merged_params = merged_task_vector.combine_with_pretrained_model(
         pretrained_model=merged_model,
        scaling_coefficient=scaling_coefficient
    )

    # 把合并后的参数字典拷贝回原模型
    for param_name, param_value in merged_model.named_parameters():
        if param_name in merged_params:
            param_value.data.copy_(merged_params[param_name])

    return merged_model

# 基本参数

In [107]:
# base model
base_model_dir = "models/Qwen/Qwen2___5-Math-7B/"

# 合并模型，可以多个，逗号分隔
models_to_merge_list = "models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B/"

# 输出模型路径
output_dir = "models/merged_qwen_math_7B"

# 合并系数
scaling_coefficient = 0.7
device = "cpu"

# 加载模型

In [119]:
# 基础模型
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_dir, torch_dtype=torch.bfloat16
).to(device)

# 待合并的模型
candidate_models = []
for model_to_merge in models_to_merge_list.split(","):
    candidate_models.append(
        AutoModelForCausalLM.from_pretrained(
            model_to_merge, torch_dtype=torch.bfloat16
        ).to(device)
    )

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [120]:
print(base_model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [121]:
print(candidate_models[0])

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

# 合并模型

In [114]:
merged_model = merging_models(
    merged_model=base_model,
    models_to_merge=candidate_models,
    scaling_coefficient=scaling_coefficient
)

In [115]:
print(merged_model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [116]:
# 保存合并后的模型
print(f"Saving model to {output_dir}")
merged_model = merged_model.to(torch.bfloat16)
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('models/merged_qwen_math_7B/tokenizer_config.json',
 'models/merged_qwen_math_7B/special_tokens_map.json',
 'models/merged_qwen_math_7B/vocab.json',
 'models/merged_qwen_math_7B/merges.txt',
 'models/merged_qwen_math_7B/added_tokens.json',
 'models/merged_qwen_math_7B/tokenizer.json')

In [122]:
sft_param_dict = {param_name: param_value
        for param_name, param_value in candidate_models[0].named_parameters()}
merged_param_dict = {param_name: param_value
        for param_name, param_value in merged_model.named_parameters()}

In [123]:
print(sft_param_dict.keys())

dict_keys(['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.post_a

In [124]:
sft_param_dict["model.layers.27.self_attn.q_proj.weight"]

Parameter containing:
tensor([[-0.0090,  0.0223, -0.0084,  ..., -0.0303,  0.0260,  0.0106],
        [ 0.0148, -0.0405, -0.0203,  ..., -0.0060, -0.0117, -0.0032],
        [ 0.0156, -0.0189, -0.0310,  ..., -0.0060, -0.0031,  0.0111],
        ...,
        [-0.0469,  0.0007,  0.0303,  ..., -0.0115, -0.0723, -0.0781],
        [ 0.0425,  0.0020,  0.0039,  ...,  0.0396, -0.1035,  0.0767],
        [ 0.0302,  0.0067, -0.0187,  ...,  0.0535,  0.0089,  0.0297]],
       dtype=torch.bfloat16, requires_grad=True)

In [125]:
merged_param_dict["model.layers.27.self_attn.q_proj.weight"]

Parameter containing:
tensor([[-0.0091,  0.0223, -0.0073,  ..., -0.0292,  0.0266,  0.0118],
        [ 0.0134, -0.0398, -0.0193,  ..., -0.0055, -0.0100, -0.0051],
        [ 0.0161, -0.0195, -0.0315,  ..., -0.0055, -0.0025,  0.0106],
        ...,
        [-0.0471, -0.0004,  0.0300,  ..., -0.0128, -0.0713, -0.0781],
        [ 0.0408, -0.0010,  0.0028,  ...,  0.0403, -0.1030,  0.0781],
        [ 0.0288,  0.0066, -0.0177,  ...,  0.0557,  0.0106,  0.0304]],
       dtype=torch.bfloat16, requires_grad=True)